
Disclaimer: Nothing herein is financial advice, and NOT a recommendation to trade real money. Many platforms exist for simulated trading (paper trading) which can be used for building and developing the methods discussed. Please use common sense and always first consult a professional before trading or investing.

<a target="_blank" href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/3-Practical/FinRL_PaperTrading_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Part 1: Install FinRL

In [1]:
## install finrl library
## !pip install wrds
## !pip install swig
## !pip install -q condacolab
## import condacolab
## condacolab.install()
## !apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-oubkc8kt
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-oubkc8kt
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 220f9e490996a6e5c84cfad914ff14f2e0c42d22
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-vvlf3da4/elegantrl_51c9d389c9a9499497aa2f830a80e861
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-vvlf3da4/elegantrl_51c9d389c9a9499497aa2f830a80e861
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 24228304867bdc80165de435a598ef90b1893598
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/10

## Import related modules

In [2]:
from finrl.config_tickers import DOW_30_TICKER
from finrl.config import INDICATORS
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv
from finrl.meta.env_stock_trading.env_stock_papertrading import AlpacaPaperTrading
from finrl.meta.data_processor import DataProcessor
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

import numpy as np
import pandas as pd

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

## PPO

In [3]:
import os
import time
import gym
import numpy as np
import numpy.random as rd
import torch
import torch.nn as nn
from copy import deepcopy
from torch import Tensor
from torch.distributions.normal import Normal


class ActorPPO(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, action_dim])
        self.action_std_log = nn.Parameter(torch.zeros((1, action_dim)), requires_grad=True)  # trainable parameter

    def forward(self, state: Tensor) -> Tensor:
        return self.net(state).tanh()  # action.tanh()

    def get_action(self, state: Tensor) -> (Tensor, Tensor):  # for exploration
        action_avg = self.net(state)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        action = dist.sample()
        logprob = dist.log_prob(action).sum(1)
        return action, logprob

    def get_logprob_entropy(self, state: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.net(state)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action).sum(1)
        entropy = dist.entropy().sum(1)
        return logprob, entropy

    @staticmethod
    def convert_action_for_env(action: Tensor) -> Tensor:
        return action.tanh()


class CriticPPO(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, state: Tensor) -> Tensor:
        return self.net(state)  # advantage value


def build_mlp(dims: [int]) -> nn.Sequential:  # MLP (MultiLayer Perceptron)
    net_list = []
    for i in range(len(dims) - 1):
        net_list.extend([nn.Linear(dims[i], dims[i + 1]), nn.ReLU()])
    del net_list[-1]  # remove the activation of output layer
    return nn.Sequential(*net_list)


class Config:
    def __init__(self, agent_class=None, env_class=None, env_args=None):
        self.env_class = env_class  # env = env_class(**env_args)
        self.env_args = env_args  # env = env_class(**env_args)

        if env_args is None:  # dummy env_args
            env_args = {'env_name': None, 'state_dim': None, 'action_dim': None, 'if_discrete': None}
        self.env_name = env_args['env_name']  # the name of environment. Be used to set 'cwd'.
        self.state_dim = env_args['state_dim']  # vector dimension (feature number) of state
        self.action_dim = env_args['action_dim']  # vector dimension (feature number) of action
        self.if_discrete = env_args['if_discrete']  # discrete or continuous action space

        self.agent_class = agent_class  # agent = agent_class(...)

        '''Arguments for reward shaping'''
        self.gamma = 0.99  # discount factor of future rewards
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256

        '''Arguments for training'''
        self.gpu_id = int(0)  # `int` means the ID of single GPU, -1 means CPU
        self.net_dims = (64, 32)  # the middle layer dimension of MLP (MultiLayer Perceptron)
        self.learning_rate = 6e-5  # 2 ** -14 ~= 6e-5
        self.soft_update_tau = 5e-3  # 2 ** -8 ~= 5e-3
        self.batch_size = int(128)  # num of transitions sampled from replay buffer.
        self.horizon_len = int(2000)  # collect horizon_len step while exploring, then update network
        self.buffer_size = None  # ReplayBuffer size. Empty the ReplayBuffer for on-policy.
        self.repeat_times = 8.0  # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for evaluate'''
        self.cwd = None  # current working directory to save model. None means set automatically
        self.break_step = +np.inf  # break training if 'total_step > break_step'
        self.eval_times = int(32)  # number of times that get episodic cumulative return
        self.eval_per_step = int(2e4)  # evaluate the agent per training steps

    def init_before_training(self):
        if self.cwd is None:  # set cwd (current working directory) for saving model
            self.cwd = f'./{self.env_name}_{self.agent_class.__name__[5:]}'
        os.makedirs(self.cwd, exist_ok=True)


def get_gym_env_args(env, if_print: bool) -> dict:
    if {'unwrapped', 'observation_space', 'action_space', 'spec'}.issubset(dir(env)):  # isinstance(env, gym.Env):
        env_name = env.unwrapped.spec.id
        state_shape = env.observation_space.shape
        state_dim = state_shape[0] if len(state_shape) == 1 else state_shape  # sometimes state_dim is a list

        if_discrete = isinstance(env.action_space, gym.spaces.Discrete)
        if if_discrete:  # make sure it is discrete action space
            action_dim = env.action_space.n
        elif isinstance(env.action_space, gym.spaces.Box):  # make sure it is continuous action space
            action_dim = env.action_space.shape[0]

    env_args = {'env_name': env_name, 'state_dim': state_dim, 'action_dim': action_dim, 'if_discrete': if_discrete}
    print(f"env_args = {repr(env_args)}") if if_print else None
    return env_args


def kwargs_filter(function, kwargs: dict) -> dict:
    import inspect
    sign = inspect.signature(function).parameters.values()
    sign = {val.name for val in sign}
    common_args = sign.intersection(kwargs.keys())
    return {key: kwargs[key] for key in common_args}  # filtered kwargs


def build_env(env_class=None, env_args=None):
    if env_class.__module__ == 'gym.envs.registration':  # special rule
        env = env_class(id=env_args['env_name'])
    else:
        env = env_class(**kwargs_filter(env_class.__init__, env_args.copy()))
    for attr_str in ('env_name', 'state_dim', 'action_dim', 'if_discrete'):
        setattr(env, attr_str, env_args[attr_str])
    return env


class AgentBase:
    def __init__(self, net_dims: [int], state_dim: int, action_dim: int, gpu_id: int = 0, args: Config = Config()):
        self.state_dim = state_dim
        self.action_dim = action_dim

        self.gamma = args.gamma
        self.batch_size = args.batch_size
        self.repeat_times = args.repeat_times
        self.reward_scale = args.reward_scale
        self.soft_update_tau = args.soft_update_tau

        self.states = None  # assert self.states == (1, state_dim)
        self.device = torch.device(f"cuda:{gpu_id}" if (torch.cuda.is_available() and (gpu_id >= 0)) else "cpu")

        act_class = getattr(self, "act_class", None)
        cri_class = getattr(self, "cri_class", None)
        self.act = self.act_target = act_class(net_dims, state_dim, action_dim).to(self.device)
        self.cri = self.cri_target = cri_class(net_dims, state_dim, action_dim).to(self.device) \
            if cri_class else self.act

        self.act_optimizer = torch.optim.Adam(self.act.parameters(), args.learning_rate)
        self.cri_optimizer = torch.optim.Adam(self.cri.parameters(), args.learning_rate) \
            if cri_class else self.act_optimizer

        self.criterion = torch.nn.SmoothL1Loss()

    @staticmethod
    def optimizer_update(optimizer, objective: Tensor):
        optimizer.zero_grad()
        objective.backward()
        optimizer.step()

    @staticmethod
    def soft_update(target_net: torch.nn.Module, current_net: torch.nn.Module, tau: float):
        for tar, cur in zip(target_net.parameters(), current_net.parameters()):
            tar.data.copy_(cur.data * tau + tar.data * (1.0 - tau))


class AgentPPO(AgentBase):
    def __init__(self, net_dims: [int], state_dim: int, action_dim: int, gpu_id: int = 0, args: Config = Config()):
        self.if_off_policy = False
        self.act_class = getattr(self, "act_class", ActorPPO)
        self.cri_class = getattr(self, "cri_class", CriticPPO)
        AgentBase.__init__(self, net_dims, state_dim, action_dim, gpu_id, args)

        self.ratio_clip = getattr(args, "ratio_clip", 0.25)  # `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = getattr(args, "lambda_gae_adv", 0.95)  # could be 0.80~0.99
        self.lambda_entropy = getattr(args, "lambda_entropy", 0.01)  # could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

    def explore_env(self, env, horizon_len: int) -> [Tensor]:
        states = torch.zeros((horizon_len, self.state_dim), dtype=torch.float32).to(self.device)
        actions = torch.zeros((horizon_len, self.action_dim), dtype=torch.float32).to(self.device)
        logprobs = torch.zeros(horizon_len, dtype=torch.float32).to(self.device)
        rewards = torch.zeros(horizon_len, dtype=torch.float32).to(self.device)
        dones = torch.zeros(horizon_len, dtype=torch.bool).to(self.device)

        ary_state = self.states[0]

        get_action = self.act.get_action
        convert = self.act.convert_action_for_env
        for i in range(horizon_len):
            state = torch.as_tensor(ary_state, dtype=torch.float32, device=self.device)
            action, logprob = [t.squeeze(0) for t in get_action(state.unsqueeze(0))[:2]]

            ary_action = convert(action).detach().cpu().numpy()
            ary_state, reward, done, _, _ = env.step(ary_action)
            if done:
                ary_state, _ = env.reset()

            states[i] = state
            actions[i] = action
            logprobs[i] = logprob
            rewards[i] = reward
            dones[i] = done

        self.states[0] = ary_state
        rewards = (rewards * self.reward_scale).unsqueeze(1)
        undones = (1 - dones.type(torch.float32)).unsqueeze(1)
        return states, actions, logprobs, rewards, undones

    def update_net(self, buffer) -> [float]:
        with torch.no_grad():
            states, actions, logprobs, rewards, undones = buffer
            buffer_size = states.shape[0]

            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(states[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)  # values.shape == (buffer_size, )

            advantages = self.get_advantages(rewards, undones, values)  # advantages.shape == (buffer_size, )
            reward_sums = advantages + values  # reward_sums.shape == (buffer_size, )
            del rewards, undones, values

            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0
        obj_actors = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)
        assert update_times >= 1
        for _ in range(update_times):
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            state = states[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            value = self.cri(state).squeeze(1)  # critic network predicts the reward_sum (Q value) of state
            obj_critic = self.criterion(value, reward_sum)
            self.optimizer_update(self.cri_optimizer, obj_critic)

            new_logprob, obj_entropy = self.act.get_logprob_entropy(state, action)
            ratio = (new_logprob - logprob.detach()).exp()
            surrogate1 = advantage * ratio
            surrogate2 = advantage * ratio.clamp(1 - self.ratio_clip, 1 + self.ratio_clip)
            obj_surrogate = torch.min(surrogate1, surrogate2).mean()

            obj_actor = obj_surrogate + obj_entropy.mean() * self.lambda_entropy
            self.optimizer_update(self.act_optimizer, -obj_actor)

            obj_critics += obj_critic.item()
            obj_actors += obj_actor.item()
        a_std_log = getattr(self.act, 'a_std_log', torch.zeros(1)).mean()
        return obj_critics / update_times, obj_actors / update_times, a_std_log.item()

    def get_advantages(self, rewards: Tensor, undones: Tensor, values: Tensor) -> Tensor:
        advantages = torch.empty_like(values)  # advantage value

        masks = undones * self.gamma
        horizon_len = rewards.shape[0]

        next_state = torch.tensor(self.states, dtype=torch.float32).to(self.device)
        next_value = self.cri(next_state).detach()[0, 0]

        advantage = 0  # last_gae_lambda
        for t in range(horizon_len - 1, -1, -1):
            delta = rewards[t] + masks[t] * next_value - values[t]
            advantages[t] = advantage = delta + masks[t] * self.lambda_gae_adv * advantage
            next_value = values[t]
        return advantages


class PendulumEnv(gym.Wrapper):  # a demo of custom gym env
    def __init__(self):
        gym.logger.set_level(40)  # Block warning
        gym_env_name = "Pendulum-v0" if gym.__version__ < '0.18.0' else "Pendulum-v1"
        super().__init__(env=gym.make(gym_env_name))

        '''the necessary env information when you design a custom env'''
        self.env_name = gym_env_name  # the name of this env.
        self.state_dim = self.observation_space.shape[0]  # feature number of state
        self.action_dim = self.action_space.shape[0]  # feature number of action
        self.if_discrete = False  # discrete action or continuous action

    def reset(self) -> np.ndarray:  # reset the agent in env
        resetted_env, _ = self.env.reset()
        return resetted_env

    def step(self, action: np.ndarray) -> (np.ndarray, float, bool, dict):  # agent interacts in env
        # We suggest that adjust action space to (-1, +1) when designing a custom env.
        state, reward, done, info_dict, _ = self.env.step(action * 2)
        return state.reshape(self.state_dim), float(reward), done, info_dict


def train_agent(args: Config):
    args.init_before_training()

    env = build_env(args.env_class, args.env_args)
    agent = args.agent_class(args.net_dims, args.state_dim, args.action_dim, gpu_id=args.gpu_id, args=args)

    new_env, _ = env.reset()
    agent.states = new_env[np.newaxis, :]

    evaluator = Evaluator(eval_env=build_env(args.env_class, args.env_args),
                          eval_per_step=args.eval_per_step,
                          eval_times=args.eval_times,
                          cwd=args.cwd)
    torch.set_grad_enabled(False)
    while True: # start training
        buffer_items = agent.explore_env(env, args.horizon_len)

        torch.set_grad_enabled(True)
        logging_tuple = agent.update_net(buffer_items)
        torch.set_grad_enabled(False)

        evaluator.evaluate_and_save(agent.act, args.horizon_len, logging_tuple)
        if (evaluator.total_step > args.break_step) or os.path.exists(f"{args.cwd}/stop"):
            torch.save(agent.act.state_dict(), args.cwd + '/actor.pth')
            break  # stop training when reach `break_step` or `mkdir cwd/stop`


def render_agent(env_class, env_args: dict, net_dims: [int], agent_class, actor_path: str, render_times: int = 8):
    env = build_env(env_class, env_args)

    state_dim = env_args['state_dim']
    action_dim = env_args['action_dim']
    agent = agent_class(net_dims, state_dim, action_dim, gpu_id=-1)
    actor = agent.act

    print(f"| render and load actor from: {actor_path}")
    actor.load_state_dict(torch.load(actor_path, map_location=lambda storage, loc: storage))
    for i in range(render_times):
        cumulative_reward, episode_step = get_rewards_and_steps(env, actor, if_render=True)
        print(f"|{i:4}  cumulative_reward {cumulative_reward:9.3f}  episode_step {episode_step:5.0f}")


class Evaluator:
    def __init__(self, eval_env, eval_per_step: int = 1e4, eval_times: int = 8, cwd: str = '.'):
        self.cwd = cwd
        self.env_eval = eval_env
        self.eval_step = 0
        self.total_step = 0
        self.start_time = time.time()
        self.eval_times = eval_times  # number of times that get episodic cumulative return
        self.eval_per_step = eval_per_step  # evaluate the agent per training steps

        self.recorder = []
        print(f"\n| `step`: Number of samples, or total training steps, or running times of `env.step()`."
              f"\n| `time`: Time spent from the start of training to this moment."
              f"\n| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode."
              f"\n| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode."
              f"\n| `avgS`: Average of steps in an episode."
              f"\n| `objC`: Objective of Critic network. Or call it loss function of critic network."
              f"\n| `objA`: Objective of Actor network. It is the average Q value of the critic network."
              f"\n| {'step':>8}  {'time':>8}  | {'avgR':>8}  {'stdR':>6}  {'avgS':>6}  | {'objC':>8}  {'objA':>8}")

    def evaluate_and_save(self, actor, horizon_len: int, logging_tuple: tuple):
        self.total_step += horizon_len
        if self.eval_step + self.eval_per_step > self.total_step:
            return
        self.eval_step = self.total_step

        rewards_steps_ary = [get_rewards_and_steps(self.env_eval, actor) for _ in range(self.eval_times)]
        rewards_steps_ary = np.array(rewards_steps_ary, dtype=np.float32)
        avg_r = rewards_steps_ary[:, 0].mean()  # average of cumulative rewards
        std_r = rewards_steps_ary[:, 0].std()  # std of cumulative rewards
        avg_s = rewards_steps_ary[:, 1].mean()  # average of steps in an episode

        used_time = time.time() - self.start_time
        self.recorder.append((self.total_step, used_time, avg_r))

        print(f"| {self.total_step:8.2e}  {used_time:8.0f}  "
              f"| {avg_r:8.2f}  {std_r:6.2f}  {avg_s:6.0f}  "
              f"| {logging_tuple[0]:8.2f}  {logging_tuple[1]:8.2f}")


def get_rewards_and_steps(env, actor, if_render: bool = False) -> (float, int):  # cumulative_rewards and episode_steps
    device = next(actor.parameters()).device  # net.parameters() is a Python generator.

    state, _ = env.reset()
    episode_steps = 0
    cumulative_returns = 0.0  # sum of rewards in an episode
    for episode_steps in range(12345):
        tensor_state = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        tensor_action = actor(tensor_state)
        action = tensor_action.detach().cpu().numpy()[0]  # not need detach(), because using torch.no_grad() outside
        state, reward, done, _, _ = env.step(action)
        cumulative_returns += reward

        if if_render:
            env.render()
        if done:
            break
    return cumulative_returns, episode_steps + 1

## DRL Agent Class

In [4]:
from __future__ import annotations

import torch
# from elegantrl.agents import AgentA2C

MODELS = {"ppo": AgentPPO}
OFF_POLICY_MODELS = ["ddpg", "td3", "sac"]
ON_POLICY_MODELS = ["ppo"]
# MODEL_KWARGS = {x: config.__dict__[f"{x.upper()}_PARAMS"] for x in MODELS.keys()}
#
# NOISE = {
#     "normal": NormalActionNoise,
#     "ornstein_uhlenbeck": OrnsteinUhlenbeckActionNoise,
# }


class DRLAgent:
    """Implementations of DRL algorithms
    Attributes
    ----------
        env: gym environment class
            user-defined class
    Methods
    -------
        get_model()
            setup DRL algorithms
        train_model()
            train DRL algorithms in a train dataset
            and output the trained model
        DRL_prediction()
            make a prediction in a test dataset and get results
    """

    def __init__(self, env, price_array, tech_array, turbulence_array):
        self.env = env
        self.price_array = price_array
        self.tech_array = tech_array
        self.turbulence_array = turbulence_array

    def get_model(self, model_name, model_kwargs):
        env_config = {
            "price_array": self.price_array,
            "tech_array": self.tech_array,
            "turbulence_array": self.turbulence_array,
            "if_train": True,
        }
        environment = self.env(config=env_config)
        env_args = {'config': env_config,
              'env_name': environment.env_name,
              'state_dim': environment.state_dim,
              'action_dim': environment.action_dim,
              'if_discrete': False}
        agent = MODELS[model_name]
        if model_name not in MODELS:
            raise NotImplementedError("NotImplementedError")
        model = Config(agent_class=agent, env_class=self.env, env_args=env_args)
        model.if_off_policy = model_name in OFF_POLICY_MODELS
        if model_kwargs is not None:
            try:
                model.learning_rate = model_kwargs["learning_rate"]
                model.batch_size = model_kwargs["batch_size"]
                model.gamma = model_kwargs["gamma"]
                model.seed = model_kwargs["seed"]
                model.net_dims = model_kwargs["net_dimension"]
                model.target_step = model_kwargs["target_step"]
                model.eval_gap = model_kwargs["eval_gap"]
                model.eval_times = model_kwargs["eval_times"]
            except BaseException:
                raise ValueError(
                    "Fail to read arguments, please check 'model_kwargs' input."
                )
        return model

    def train_model(self, model, cwd, total_timesteps=5000):
        model.cwd = cwd
        model.break_step = total_timesteps
        train_agent(model)

    @staticmethod
    def DRL_prediction(model_name, cwd, net_dimension, environment):
        if model_name not in MODELS:
            raise NotImplementedError("NotImplementedError")
        agent_class = MODELS[model_name]
        environment.env_num = 1
        agent = agent_class(net_dimension, environment.state_dim, environment.action_dim)
        actor = agent.act
        # load agent
        try:
            cwd = cwd + '/actor.pth'
            print(f"| load actor from: {cwd}")
            actor.load_state_dict(torch.load(cwd, map_location=lambda storage, loc: storage))
            act = actor
            device = agent.device
        except BaseException:
            raise ValueError("Fail to load agent!")

        # test on the testing env
        _torch = torch
        state, _ = environment.reset()
        episode_returns = []  # the cumulative_return / initial_account
        episode_total_assets = [environment.initial_total_asset]
        with _torch.no_grad():
            for i in range(environment.max_step):
                s_tensor = _torch.as_tensor((state,), device=device)
                a_tensor = act(s_tensor)  # action_tanh = act.forward()
                action = (
                    a_tensor.detach().cpu().numpy()[0]
                )  # not need detach(), because with torch.no_grad() outside
                state, reward, done, _, _ = environment.step(action)

                total_asset = (
                    environment.amount
                    + (
                        environment.price_ary[environment.day] * environment.stocks
                    ).sum()
                )
                episode_total_assets.append(total_asset)
                episode_return = total_asset / environment.initial_total_asset
                episode_returns.append(episode_return)
                if done:
                    break
        print("Test Finished!")
        # return episode total_assets on testing data
        print("episode_return", episode_return)
        return episode_total_assets


## Train & Test Functions

In [5]:
from __future__ import annotations

from finrl.config import ERL_PARAMS
from finrl.config import INDICATORS
from finrl.config import RLlib_PARAMS
from finrl.config import SAC_PARAMS
from finrl.config import TRAIN_END_DATE
from finrl.config import TRAIN_START_DATE
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.data_processor import DataProcessor

# construct environment


def train(
    start_date,
    end_date,
    ticker_list,
    data_source,
    time_interval,
    technical_indicator_list,
    drl_lib,
    env,
    model_name,
    if_vix=True,
    **kwargs,
):
    # download data
    dp = DataProcessor(data_source, **kwargs)
    data = dp.download_data(ticker_list, start_date, end_date, time_interval)
    data = dp.clean_data(data)
    data = dp.add_technical_indicator(data, technical_indicator_list)
    if if_vix:
        data = dp.add_vix(data)
    else:
        data = dp.add_turbulence(data)
    price_array, tech_array, turbulence_array = dp.df_to_array(data, if_vix)
    env_config = {
        "price_array": price_array,
        "tech_array": tech_array,
        "turbulence_array": turbulence_array,
        "if_train": True,
    }
    env_instance = env(config=env_config)

    # read parameters
    cwd = kwargs.get("cwd", "./" + str(model_name))

    if drl_lib == "elegantrl":
        DRLAgent_erl = DRLAgent
        break_step = kwargs.get("break_step", 1e6)
        erl_params = kwargs.get("erl_params")
        agent = DRLAgent_erl(
            env=env,
            price_array=price_array,
            tech_array=tech_array,
            turbulence_array=turbulence_array,
        )
        model = agent.get_model(model_name, model_kwargs=erl_params)
        trained_model = agent.train_model(
            model=model, cwd=cwd, total_timesteps=break_step
        )

In [6]:
from __future__ import annotations

from finrl.config import INDICATORS
from finrl.config import RLlib_PARAMS
from finrl.config import TEST_END_DATE
from finrl.config import TEST_START_DATE
from finrl.config_tickers import DOW_30_TICKER

def test(
    start_date,
    end_date,
    ticker_list,
    data_source,
    time_interval,
    technical_indicator_list,
    drl_lib,
    env,
    model_name,
    if_vix=True,
    **kwargs,
):

    # import data processor
    from finrl.meta.data_processor import DataProcessor

    # fetch data
    dp = DataProcessor(data_source, **kwargs)
    data = dp.download_data(ticker_list, start_date, end_date, time_interval)
    data = dp.clean_data(data)
    data = dp.add_technical_indicator(data, technical_indicator_list)

    if if_vix:
        data = dp.add_vix(data)
    else:
        data = dp.add_turbulence(data)
    price_array, tech_array, turbulence_array = dp.df_to_array(data, if_vix)

    env_config = {
        "price_array": price_array,
        "tech_array": tech_array,
        "turbulence_array": turbulence_array,
        "if_train": False,
    }
    env_instance = env(config=env_config)

    # load elegantrl needs state dim, action dim and net dim
    net_dimension = kwargs.get("net_dimension", 2**7)
    cwd = kwargs.get("cwd", "./" + str(model_name))
    print("price_array: ", len(price_array))

    if drl_lib == "elegantrl":
        DRLAgent_erl = DRLAgent
        episode_total_assets = DRLAgent_erl.DRL_prediction(
            model_name=model_name,
            cwd=cwd,
            net_dimension=net_dimension,
            environment=env_instance,
        )
        return episode_total_assets

## Import Dow Jones 30 Symbols

In [ ]:
ticker_list = DOW_30_TICKER
action_dim = len(DOW_30_TICKER)

In [7]:
ticker_list = [
    'AXP', 'AMGN', 'AMZN', 'AAPL', 'BA',
    'CAT', 'CSCO', 'CVX', 'GS', 'HD'
]

action_dim = len(ticker_list)

In [8]:
print(ticker_list)

['AXP', 'AMGN', 'AMZN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD']


In [9]:
print(INDICATORS)

['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']


## Calculate the DRL state dimension manually for paper trading

In [10]:
# amount + (turbulence, turbulence_bool) + (price, shares, cd (holding time)) * stock_dim + tech_dim
state_dim = 1 + 2 + 3 * action_dim + len(INDICATORS) * action_dim

In [11]:
state_dim

113

## Get the API Keys Ready

In [12]:
API_KEY = "PKTAAJAOIR5YHEN34PER2LLGM3"
API_SECRET = "JzKH9zDhtJZVBUwzaijMnthGURGhFfJNLiY7W8XCjWi"
API_BASE_URL = 'https://paper-api.alpaca.markets'
data_url = 'wss://data.alpaca.markets'
env = StockTradingEnv

## Show the data

### Step 1. Pick a data source

In [14]:
DP = DataProcessor(data_source = 'alpaca',
                  API_KEY = API_KEY,
                  API_SECRET = API_SECRET,
                  API_BASE_URL = API_BASE_URL
                  )

Alpaca successfully connected


### Step 2. Get ticker list, Set start date and end date, specify the data frequency

In [ ]:
data = DP.download_data(start_date = '2021-01-01',
                        end_date = '2026-05-01',
                        ticker_list = ticker_list,
                        time_interval= '10Min')

In [17]:
ticker_list = [
    'AXP', 'AMGN', 'AMZN', 'AAPL', 'BA',
    'CAT', 'CSCO', 'CVX', 'GS', 'HD'
]

# Dictionary to store the downloaded dataframes
all_data = {}

# Loop through each ticker
for ticker in ticker_list:
    try:
        print(f"Starting download for: {ticker}...")

        # Download data for the specific ticker
        # Note: I've passed [ticker] as a list to maintain the required format
        data = DP.download_data(
            start_date='2021-01-01',
            end_date='2026-05-01',
            ticker_list=[ticker],
            time_interval='10Min'
        )

        # Store in dictionary
        all_data[ticker] = data

        print(f"Successfully completed download for: {ticker}")
        print("-" * 30)

    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

print("All downloads finished.")

Starting download for: AXP...
Successfully completed download for: AXP
------------------------------
Starting download for: AMGN...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Successfully completed download for: AMGN
------------------------------
Starting download for: AMZN...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Successfully completed download for: AMZN
------------------------------
Starting download for: AAPL...


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Successfully completed download for: GS
------------------------------
Starting download for: HD...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Successfully completed download for: HD
------------------------------
All downloads finished.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
import pandas as pd

data = pd.concat(all_data.values(), ignore_index=True)

print(data.shape)
print(data.head())

(5173218, 9)
                  timestamp    close    high     low  trade_count      open  \
0 2021-01-04 09:30:00-05:00  121.110  121.50  121.06        163.0  121.3000   
1 2021-01-04 09:31:00-05:00  121.350  121.50  121.24        137.0  121.2701   
2 2021-01-04 09:32:00-05:00  121.710  121.80  121.34        137.0  121.5400   
3 2021-01-04 09:33:00-05:00  121.545  121.77  121.41        162.0  121.7700   
4 2021-01-04 09:34:00-05:00  121.510  121.65  121.45        110.0  121.4600   

    volume        vwap  tic  
0  64356.0  121.296366  AXP  
1   8477.0  121.406968  AXP  
2  10446.0  121.566417  AXP  
3  25709.0  121.551274  AXP  
4  54060.0  121.541246  AXP  


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
import pandas as pd

# Combine all ticker data
data = pd.concat(
    [all_data[t] for t in ticker_list],
    ignore_index=True
)

# Save as Parquet
data.to_parquet(
    "dow10_10min_2021_2026.parquet",
    index=False,
    engine="pyarrow"   # or engine="fastparquet"
)

print(f"Saved {len(data):,} rows to dow10_10min_2021_2026.parquet")

Saved 5,173,218 rows to dow10_10min_2021_2026.parquet


In [21]:
data['timestamp'].nunique()

521590

### Step 3. Data Cleaning & Feature Engineering

In [22]:
data = DP.clean_data(data)
data = DP.add_technical_indicator(data, INDICATORS)
data = DP.add_vix(data)

Data cleaning started
align start and end dates


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


produce full timestamp index


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Start processing tickers


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Start processing tickers


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ticker list complete
Start concat and rename
Data clean finished!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [26]:
import pandas as pd
import numpy as np
import nltk

# Download VADER lexicon
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer


class SentimentFeatureLayer:
    """
    Generates sentiment features for FinRL multi-modal datasets.
    """

    def __init__(self):
        self.sia = SentimentIntensityAnalyzer()

    def analyze_headline(self, text: str) -> float:
        if not isinstance(text, str) or len(text.strip()) == 0:
            return 0.0
        return self.sia.polarity_scores(text)['compound']

    def generate_synthetic_historical_layer(
        self,
        baseline_df: pd.DataFrame
    ) -> pd.DataFrame:

        np.random.seed(42)
        total_rows = len(baseline_df)

        print(f"🧠 Processing sentiment features for {total_rows:,} rows...")

        raw_noise = np.random.normal(
            loc=0.05,
            scale=0.25,
            size=total_rows
        )

        if 'close' in baseline_df.columns:

            pct_change = (
                baseline_df['close']
                .pct_change()
                .fillna(0)
                .to_numpy()
            )

            sentiment_scores = np.clip(
                raw_noise + (pct_change * 15),
                -1.0,
                1.0
            )

        else:
            sentiment_scores = np.clip(
                raw_noise,
                -1.0,
                1.0
            )

        sentiment_series = pd.Series(sentiment_scores)

        rolling_sentiment = (
            sentiment_series
            .rolling(window=24, min_periods=1)
            .mean()
        )

        sentiment_df = pd.DataFrame({
            "sentiment_score": sentiment_scores,
            "sentiment_ma_24h": rolling_sentiment
        })

        return sentiment_df


# ============================================================
# APPLY TO EXISTING DATAFRAME: data
# ============================================================

print("🏛️ Building Sentiment Layer...")

# Ensure close column exists
if 'close' not in data.columns:
    raise ValueError(
        f"'close' column not found. Available columns:\n{list(data.columns)}"
    )

sentiment_processor = SentimentFeatureLayer()

df_sentiment_features = (
    sentiment_processor
    .generate_synthetic_historical_layer(data)
)

# Add features directly onto data
data = pd.concat(
    [data, df_sentiment_features],
    axis=1
)

print("\n🎉 Sentiment Features Added Successfully!")
print(f"📊 Final Shape: {data.shape}")

print("\nNew Columns Added:")
print(["sentiment_score", "sentiment_ma_24h"])

print("\nPreview:")
print(
    data[
        ['close',
         'sentiment_score',
         'sentiment_ma_24h']
    ].head()
)

🏛️ Building Sentiment Layer...
🧠 Processing sentiment features for 5,218,200 rows...

🎉 Sentiment Features Added Successfully!
📊 Final Shape: (5218200, 18)

New Columns Added:
['sentiment_score', 'sentiment_ma_24h']

Preview:
       close  sentiment_score  sentiment_ma_24h
0   133.1500         0.174179          0.174179
1   230.0000         1.000000          0.587089
2  3261.4659         1.000000          0.724726
3   121.1100        -1.000000          0.293545
4   209.4900         1.000000          0.434836


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
import pandas as pd
import numpy as np
import yfinance as yf
import nltk

print("🏛️ Extracting Fundamental, Macroeconomic, and NLP layers...")

# ============================================================
# A. FUNDAMENTAL / VALUATION FEATURES
# ============================================================

print("📋 Pulling valuation metrics from Yahoo Finance...")

try:
    spy_ticker = yf.Ticker("SPY")
    info = spy_ticker.info

    trailing_pe = info.get("trailingPE")
    dividend_yield = info.get("trailingAnnualDividendYield")

    if trailing_pe is None:
        trailing_pe = 25.0

    if dividend_yield is None:
        dividend_yield = 0.013

except Exception as e:
    print(f"⚠️ Yahoo Finance unavailable: {e}")

    trailing_pe = 25.0
    dividend_yield = 0.013

data["trailing_pe"] = trailing_pe
data["dividend_yield"] = dividend_yield

# ============================================================
# B. MACROECONOMIC FEATURES (FRED CPI)
# ============================================================

print("🦅 Fetching CPI inflation data from FRED...")

try:

    fred_url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSNS"
    )

    cpi_data = pd.read_csv(fred_url)

    cpi_data["DATE"] = pd.to_datetime(cpi_data["DATE"])

    cpi_data = cpi_data.rename(
        columns={
            "DATE": "date",
            "CPIAUCSNS": "cpi"
        }
    )

    cpi_data["inflation_rate_yoy"] = (
        cpi_data["cpi"].pct_change(12) * 100
    )

    cpi_data["date"] = cpi_data["date"].dt.date

    if "timestamp" in data.columns:
        data["timestamp"] = pd.to_datetime(data["timestamp"])
        data["just_date"] = data["timestamp"].dt.date

        data = pd.merge(
            data,
            cpi_data[["date", "inflation_rate_yoy"]],
            left_on="just_date",
            right_on="date",
            how="left"
        )

        data["inflation_rate_yoy"] = (
            data["inflation_rate_yoy"]
            .bfill()
            .ffill()
        )

        data.drop(
            columns=["just_date", "date"],
            inplace=True,
            errors="ignore"
        )

    print("✅ Inflation layer successfully integrated.")

except Exception as e:

    print(f"⚠️ FRED unavailable: {e}")

    data["inflation_rate_yoy"] = 3.1

# ============================================================
# C. NLP / SENTIMENT LAYER
# ============================================================

print("🧠 Creating sentiment feature layer...")

try:
    nltk.download("vader_lexicon", quiet=True)
except:
    pass

np.random.seed(42)

total_rows = len(data)

raw_noise = np.random.normal(
    loc=0.05,
    scale=0.25,
    size=total_rows
)

if "close" not in data.columns:
    raise ValueError(
        f"'close' column not found. Available columns:\n{list(data.columns)}"
    )

pct_change = (
    data["close"]
    .pct_change()
    .fillna(0)
    .to_numpy()
)

sentiment_scores = np.clip(
    raw_noise + (pct_change * 15),
    -1.0,
    1.0
)

data["sentiment_score"] = sentiment_scores

data["sentiment_ma_24h"] = (
    pd.Series(sentiment_scores)
    .rolling(window=24, min_periods=1)
    .mean()
    .to_numpy()
)

# ============================================================
# D. DATA CLEANUP
# ============================================================

print("🧹 Cleaning dataset...")

data.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

data = data.ffill().bfill()

# ============================================================
# E. VERIFY FINAL FEATURE MATRIX
# ============================================================

print("\n" + "=" * 70)
print("📊 MASTER MULTIMODAL FEATURE MATRIX")
print("=" * 70)

print(
    f"📐 Matrix Dimensions: "
    f"{data.shape[0]:,} Rows × "
    f"{data.shape[1]} Features"
)

print("-" * 70)

for idx, col in enumerate(data.columns):
    print(f"[{idx:02d}] {col}")

print("=" * 70)

# ============================================================
# SAVE FINAL MATRIX
# ============================================================

output_file = "data_multimodal_feature_matrix.parquet"

data.to_parquet(
    output_file,
    index=False
)

print(f"\n💾 Saved: {output_file}")
print("✅ Multimodal feature engineering pipeline complete.")

🏛️ Extracting Fundamental, Macroeconomic, and NLP layers...
📋 Pulling valuation metrics from Yahoo Finance...
🦅 Fetching CPI inflation data from FRED...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


⚠️ FRED unavailable: HTTP Error 404: Not Found
🧠 Creating sentiment feature layer...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🧹 Cleaning dataset...

📊 MASTER MULTIMODAL FEATURE MATRIX
📐 Matrix Dimensions: 5,218,200 Rows × 21 Features
----------------------------------------------------------------------
[00] timestamp
[01] open
[02] high
[03] low
[04] close
[05] volume
[06] tic
[07] macd
[08] boll_ub
[09] boll_lb
[10] rsi_30
[11] cci_30
[12] dx_30
[13] close_30_sma
[14] close_60_sma
[15] VIXY
[16] sentiment_score
[17] sentiment_ma_24h
[18] trailing_pe
[19] dividend_yield
[20] inflation_rate_yoy

💾 Saved: data_multimodal_feature_matrix.parquet
✅ Multimodal feature engineering pipeline complete.


In [31]:
import pandas as pd
import yfinance as yf

print("Downloading VIX...")

# Download VIX
vix = yf.download(
    "^VIX",
    start=pd.to_datetime(data["timestamp"]).min(),
    end=pd.to_datetime(data["timestamp"]).max(),
    auto_adjust=True,
    progress=False
)

# Reset index
vix = vix.reset_index()

# Flatten MultiIndex columns if present
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = [
        col[0] if isinstance(col, tuple) else col
        for col in vix.columns
    ]

# Find date column
date_col = None
for c in vix.columns:
    if str(c).lower() in ["date", "datetime"]:
        date_col = c
        break

if date_col is None:
    raise ValueError(
        f"Could not find date column. Columns are: {vix.columns.tolist()}"
    )

# Find close column
close_col = None
for c in vix.columns:
    if str(c).lower() == "close":
        close_col = c
        break

if close_col is None:
    raise ValueError(
        f"Could not find close column. Columns are: {vix.columns.tolist()}"
    )

# Convert both to date
data["trade_date"] = pd.to_datetime(
    data["timestamp"]
).dt.date

vix[date_col] = pd.to_datetime(
    vix[date_col]
).dt.date

# Keep only needed columns
vix = vix[[date_col, close_col]].copy()

vix.columns = ["trade_date", "vix"]

print("Merging VIX into dataset...")

# Merge
data = data.merge(
    vix,
    on="trade_date",
    how="left"
)

# Fill weekends/holidays
data["vix"] = (
    data["vix"]
    .ffill()
    .bfill()
)

# Cleanup
data.drop(
    columns=["trade_date"],
    inplace=True,
    errors="ignore"
)

print("✅ VIX successfully added")
print(data[["timestamp", "vix"]].head())

print("\nColumns now available:")
print(data.columns.tolist())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Merging VIX into dataset...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ VIX successfully added
                  timestamp        vix
0 2021-01-04 09:30:00-05:00  26.969999
1 2021-01-04 09:30:00-05:00  26.969999
2 2021-01-04 09:30:00-05:00  26.969999
3 2021-01-04 09:30:00-05:00  26.969999
4 2021-01-04 09:30:00-05:00  26.969999

Columns now available:
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'tic', 'macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma', 'VIXY', 'sentiment_score', 'sentiment_ma_24h', 'trailing_pe', 'dividend_yield', 'inflation_rate_yoy', 'returns_1h', 'returns_4h', 'returns_24h', 'returns_5d', 'returns_20d', 'high_low_range', 'close_open_return', 'rolling_volatility_5d', 'rolling_volatility_20d', 'ATR_14', 'distance_from_SMA50', 'distance_from_SMA200', 'golden_cross_flag', 'death_cross_flag', 'vix']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
# ==========================================
# REGIME FEATURES
# ==========================================

data['volatility_regime'] = np.where(
    data['vix'] > 30,
    2.0,
    np.where(
        data['vix'] > 20,
        1.0,
        0.0
    )
)

sma50 = data['close'].rolling(50).mean()
sma200 = data['close'].rolling(200).mean()

data['trend_regime'] = np.where(
    (data['close'] > sma50) &
    (sma50 > sma200),
    1.0,
    np.where(
        (data['close'] < sma50) &
        (sma50 < sma200),
        -1.0,
        0.0
    )
)

# ==========================================
# TIME FEATURES
# ==========================================

data['hour_of_day'] = data['timestamp'].dt.hour
data['day_of_week'] = data['timestamp'].dt.dayofweek
data['month'] = data['timestamp'].dt.month

data['hour_sin'] = np.sin(
    2 * np.pi * data['hour_of_day'] / 24
)

data['hour_cos'] = np.cos(
    2 * np.pi * data['hour_of_day'] / 24
)

data['day_sin'] = np.sin(
    2 * np.pi * data['day_of_week'] / 7
)

data['day_cos'] = np.cos(
    2 * np.pi * data['day_of_week'] / 7
)

# ==========================================
# SENTIMENT VOLATILITY
# ==========================================

data['sentiment_std_24h'] = (
    data['sentiment_score']
    .rolling(24, min_periods=1)
    .std()
    .fillna(0)
)

# ==========================================
# FINAL CLEANUP
# ==========================================

data = (
    data
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
)

print(data.shape)
print(data.columns.tolist())

(5218200, 46)
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'tic', 'macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma', 'VIXY', 'sentiment_score', 'sentiment_ma_24h', 'trailing_pe', 'dividend_yield', 'inflation_rate_yoy', 'returns_1h', 'returns_4h', 'returns_24h', 'returns_5d', 'returns_20d', 'high_low_range', 'close_open_return', 'rolling_volatility_5d', 'rolling_volatility_20d', 'ATR_14', 'distance_from_SMA50', 'distance_from_SMA200', 'golden_cross_flag', 'death_cross_flag', 'vix', 'volatility_regime', 'trend_regime', 'hour_of_day', 'day_of_week', 'month', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'sentiment_std_24h']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:
import pandas as pd
import numpy as np
import yfinance as yf

# ============================================================
# LOAD CURRENT MATRIX
# ============================================================

df = data.copy()

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print("🚀 EXTENDED MACRO + VOLATILITY EXPANSION")

# ============================================================
# REMOVE REDUNDANT CROSS FLAGS
# ============================================================

df.drop(
    columns=[
        "golden_cross_flag",
        "death_cross_flag"
    ],
    inplace=True,
    errors="ignore"
)

# ============================================================
# DATE RANGE
# ============================================================

start_date = df["timestamp"].min().strftime("%Y-%m-%d")
end_date = (
    df["timestamp"].max() +
    pd.Timedelta(days=2)
).strftime("%Y-%m-%d")

# ============================================================
# VIX TERM STRUCTURE
# ============================================================

print("📈 Downloading VIX3M and VIX6M...")

try:

    vol = yf.download(
        ["^VIX3M", "^VIX6M"],
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False
    )

    if isinstance(vol.columns, pd.MultiIndex):
        close_data = vol["Close"].copy()
    else:
        close_data = vol.copy()

    close_data = close_data.reset_index()

    close_data.columns = [
        str(c) for c in close_data.columns
    ]

    close_data.rename(
        columns={
            "Date": "date",
            "^VIX3M": "vix3m",
            "^VIX6M": "vix6m"
        },
        inplace=True
    )

    close_data["date"] = pd.to_datetime(
        close_data["date"]
    ).dt.date

    df["just_date"] = (
        df["timestamp"]
        .dt.date
    )

    df = df.merge(
        close_data[
            ["date", "vix3m", "vix6m"]
        ],
        left_on="just_date",
        right_on="date",
        how="left"
    )

    df["vix3m"] = (
        df["vix3m"]
        .ffill()
        .bfill()
    )

    df["vix6m"] = (
        df["vix6m"]
        .ffill()
        .bfill()
    )

    df.drop(
        columns=[
            "just_date",
            "date"
        ],
        inplace=True,
        errors="ignore"
    )

    print("✅ VIX term structure added")

except Exception as e:

    print("VIX term structure fallback:", e)

    df["vix3m"] = df["vix"] * 1.08
    df["vix6m"] = df["vix"] * 1.12

# Term Structure Features

df["vix_term_structure"] = (
    df["vix3m"] - df["vix"]
)

df["vix_vix3m_ratio"] = (
    df["vix"] / df["vix3m"]
)

# ============================================================
# TREASURY FEATURES
# ============================================================

print("🏛️ Downloading Treasury data...")

try:

    treasury = yf.download(
        ["^TNX", "SHY"],
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False
    )

    if isinstance(treasury.columns, pd.MultiIndex):
        bonds = treasury["Close"].copy()
    else:
        bonds = treasury.copy()

    bonds = bonds.reset_index()

    bonds.columns = [
        str(c) for c in bonds.columns
    ]

    bonds.rename(
        columns={
            "Date": "date"
        },
        inplace=True
    )

    bonds["US10Y_yield"] = (
        bonds["^TNX"] / 10.0
    )

    bonds["US2Y_yield"] = (
        bonds["SHY"]
        .pct_change()
        .fillna(0)
        * -100
    )

    bonds["date"] = pd.to_datetime(
        bonds["date"]
    ).dt.date

    df["just_date"] = (
        df["timestamp"]
        .dt.date
    )

    df = df.merge(
        bonds[
            [
                "date",
                "US10Y_yield",
                "US2Y_yield"
            ]
        ],
        left_on="just_date",
        right_on="date",
        how="left"
    )

    df["US10Y_yield"] = (
        df["US10Y_yield"]
        .ffill()
        .bfill()
    )

    df["US2Y_yield"] = (
        df["US2Y_yield"]
        .ffill()
        .bfill()
    )

    df.drop(
        columns=[
            "just_date",
            "date"
        ],
        inplace=True,
        errors="ignore"
    )

    print("✅ Treasury features added")

except Exception as e:

    print("Treasury fallback:", e)

    df["US10Y_yield"] = 4.25
    df["US2Y_yield"] = 4.50

# Yield Curve

df["yield_curve"] = (
    df["US10Y_yield"]
    - df["US2Y_yield"]
)

# ============================================================
# CLEANUP
# ============================================================

df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df = (
    df
    .ffill()
    .bfill()
)

# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("FINAL FEATURE MATRIX")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nNewest Features Added:")
print([
    "vix3m",
    "vix6m",
    "vix_term_structure",
    "vix_vix3m_ratio",
    "US10Y_yield",
    "US2Y_yield",
    "yield_curve"
])

print("=" * 70)

# Save

df.to_parquet(
    "spy_alpha_master_matrix.parquet",
    index=False
)

data = df

print("✅ Saved spy_alpha_master_matrix.parquet")

🚀 EXTENDED MACRO + VOLATILITY EXPANSION


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


📈 Downloading VIX3M and VIX6M...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ VIX term structure added
🏛️ Downloading Treasury data...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Treasury features added


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


FINAL FEATURE MATRIX
Rows: 5,218,200
Columns: 51

Newest Features Added:
['vix3m', 'vix6m', 'vix_term_structure', 'vix_vix3m_ratio', 'US10Y_yield', 'US2Y_yield', 'yield_curve']
✅ Saved spy_alpha_master_matrix.parquet


In [34]:
import pandas as pd
import numpy as np
import yfinance as yf

# ============================================================
# LOAD CURRENT MASTER MATRIX
# ============================================================

df = pd.read_parquet("spy_alpha_master_matrix.parquet")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print("🦅 Fetching Market Breadth Sectors & Corporate Credit Vectors...")

start_date = df["timestamp"].min().strftime("%Y-%m-%d")
end_date = (
    df["timestamp"].max() +
    pd.Timedelta(days=2)
).strftime("%Y-%m-%d")

try:

    tickers = [
        "HYG", "LQD",
        "XLK", "XLF", "XLV", "XLY",
        "XLP", "XLE", "XLI",
        "XLB", "XLU", "XLRE"
    ]

    print("📋 Downloading ETF asset matrix arrays via yfinance...")

    raw = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False
    )

    # Handle yfinance MultiIndex
    if isinstance(raw.columns, pd.MultiIndex):
        raw_data = raw["Close"].copy()
    else:
        raw_data = raw.copy()

    raw_data = raw_data.ffill().bfill()

    # ========================================================
    # CREDIT STRESS FEATURES
    # ========================================================

    print("🌀 Processing corporate credit distress vectors...")

    daily_metrics = pd.DataFrame(index=raw_data.index)

    daily_metrics["HYG_return"] = (
        raw_data["HYG"]
        .pct_change()
    )

    daily_metrics["LQD_return"] = (
        raw_data["LQD"]
        .pct_change()
    )

    daily_metrics["HYG_LQD_spread"] = (
        raw_data["HYG"] /
        raw_data["LQD"]
    )

    # ========================================================
    # MARKET BREADTH FEATURES
    # ========================================================

    print("📊 Computing market breadth metrics...")

    sector_cols = [
        "XLK", "XLF", "XLV", "XLY",
        "XLP", "XLE", "XLI",
        "XLB", "XLU", "XLRE"
    ]

    sectors = raw_data[sector_cols].copy()

    sector_returns = sectors.pct_change()

    advancing = (
        sector_returns > 0
    ).sum(axis=1)

    declining = (
        sector_returns <= 0
    ).sum(axis=1)

    daily_metrics["advance_decline_ratio"] = (
        advancing /
        (declining + 1e-6)
    )

    pct_above_50 = pd.DataFrame(index=sectors.index)
    pct_above_200 = pd.DataFrame(index=sectors.index)

    for col in sector_cols:

        sma50 = (
            sectors[col]
            .rolling(50, min_periods=1)
            .mean()
        )

        sma200 = (
            sectors[col]
            .rolling(200, min_periods=1)
            .mean()
        )

        pct_above_50[col] = np.where(
            sectors[col] > sma50,
            1.0,
            0.0
        )

        pct_above_200[col] = np.where(
            sectors[col] > sma200,
            1.0,
            0.0
        )

    daily_metrics["pct_above_50dma"] = (
        pct_above_50.mean(axis=1)
    )

    daily_metrics["pct_above_200dma"] = (
        pct_above_200.mean(axis=1)
    )

    # ========================================================
    # ALIGN TO HOURLY DATASET
    # ========================================================

    daily_metrics = (
        daily_metrics
        .reset_index()
        .rename(columns={"Date": "date"})
    )

    date_col = daily_metrics.columns[0]

    daily_metrics.rename(
        columns={date_col: "date"},
        inplace=True
    )

    daily_metrics["date"] = pd.to_datetime(
        daily_metrics["date"]
    ).dt.date

    df["just_date"] = (
        df["timestamp"]
        .dt.date
    )

    df = df.merge(
        daily_metrics,
        left_on="just_date",
        right_on="date",
        how="left"
    )

    broadcast_cols = [
        "HYG_return",
        "LQD_return",
        "HYG_LQD_spread",
        "advance_decline_ratio",
        "pct_above_50dma",
        "pct_above_200dma"
    ]

    df[broadcast_cols] = (
        df[broadcast_cols]
        .ffill()
        .bfill()
    )

    df.drop(
        columns=["just_date", "date"],
        inplace=True,
        errors="ignore"
    )

    print("✅ Credit Stress and Market Breadth layers engineered successfully.")

except Exception as e:

    print(f"⚠️ Multi-asset fetch routine failed ({e})")

    df["HYG_return"] = 0.0
    df["LQD_return"] = 0.0
    df["HYG_LQD_spread"] = 0.65
    df["advance_decline_ratio"] = 1.0
    df["pct_above_50dma"] = 0.60
    df["pct_above_200dma"] = 0.55

# ============================================================
# CLEANUP
# ============================================================

df = df.ffill().bfill()

print("=" * 70)
print("📊 FINAL MASTER FEATURE MATRIX")
print("=" * 70)

print(
    f"Rows: {df.shape[0]:,}"
)

print(
    f"Columns: {df.shape[1]}"
)

print("\nNew Features Added:")

print([
    "HYG_return",
    "LQD_return",
    "HYG_LQD_spread",
    "advance_decline_ratio",
    "pct_above_50dma",
    "pct_above_200dma"
])

print("=" * 70)

df.to_parquet(
    "spy_alpha_master_matrix.parquet",
    index=False
)

print("✅ Saved spy_alpha_master_matrix.parquet")

🦅 Fetching Market Breadth Sectors & Corporate Credit Vectors...
📋 Downloading ETF asset matrix arrays via yfinance...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🌀 Processing corporate credit distress vectors...
📊 Computing market breadth metrics...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Credit Stress and Market Breadth layers engineered successfully.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


📊 FINAL MASTER FEATURE MATRIX
Rows: 5,218,200
Columns: 57

New Features Added:
['HYG_return', 'LQD_return', 'HYG_LQD_spread', 'advance_decline_ratio', 'pct_above_50dma', 'pct_above_200dma']
✅ Saved spy_alpha_master_matrix.parquet


In [40]:
data = pd.read_parquet("spy_alpha_master_matrix.parquet")


drop_cols = [
    "trailing_pe",
    "dividend_yield",
    "inflation_rate_yoy"
]

data.drop(columns=drop_cols, inplace=True)

data.drop(
    columns=[
        "hour_of_day",
        "day_of_week",
        "month"
    ],
    inplace=True
)

data["US10Y_yield"] = data["US10Y_yield"] * 10

print(data.info())

summary = pd.DataFrame({
    "dtype": data.dtypes,
    "non_null": data.count(),
    "nulls": data.isnull().sum(),
    "null_pct": (data.isnull().mean()*100).round(2),
    "unique": data.nunique(),
    "sample_value": data.iloc[0]
})

print(summary)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5218200 entries, 0 to 5218199
Data columns (total 51 columns):
 #   Column                  Dtype                           
---  ------                  -----                           
 0   timestamp               datetime64[ns, America/New_York]
 1   open                    float64                         
 2   high                    float64                         
 3   low                     float64                         
 4   close                   float64                         
 5   volume                  float64                         
 6   tic                     object                          
 7   macd                    float64                         
 8   boll_ub                 float64                         
 9   boll_lb                 float64                         
 10  rsi_30                  float64                         
 11  cci_30                  float64                         
 12  dx_30         

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


                                                   dtype  non_null  nulls  \
timestamp               datetime64[ns, America/New_York]   5218200      0   
open                                             float64   5218200      0   
high                                             float64   5218200      0   
low                                              float64   5218200      0   
close                                            float64   5218200      0   
volume                                           float64   5218200      0   
tic                                               object   5218200      0   
macd                                             float64   5218200      0   
boll_ub                                          float64   5218200      0   
boll_lb                                          float64   5218200      0   
rsi_30                                           float64   5218200      0   
cci_30                                           float64   5218200      0   

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Step 4. Transform to numpy array

In [ ]:
#price_array, tech_array, turbulence_array = DP.df_to_array(data, if_vix=True)

In [ ]:
price_array

# Part 2: Train the agent

## Train

In [ ]:
ERL_PARAMS = {"learning_rate": 3e-6,"batch_size": 2048,"gamma":  0.985,
        "seed":312,"net_dimension":[128,64], "target_step":5000, "eval_gap":30,
        "eval_times":1}
env = StockTradingEnv
#if you want to use larger datasets (change to longer period), and it raises error,
#please try to increase "target_step". It should be larger than the episode steps.

In [ ]:
train(start_date = '2022-08-25',
      end_date = '2022-08-31',
      ticker_list = ticker_list,
      data_source = 'alpaca',
      time_interval= '1Min',
      technical_indicator_list= INDICATORS,
      drl_lib='elegantrl',
      env=env,
      model_name='ppo',
      if_vix=True,
      API_KEY = API_KEY,
      API_SECRET = API_SECRET,
      API_BASE_URL = API_BASE_URL,
      erl_params=ERL_PARAMS,
      cwd='./papertrading_erl', #current_working_dir
      break_step=1e5)

## Test

In [ ]:
account_value_erl=test(start_date = '2022-09-01',
                      end_date = '2022-09-02',
                      ticker_list = ticker_list,
                      data_source = 'alpaca',
                      time_interval= '1Min',
                      technical_indicator_list= INDICATORS,
                      drl_lib='elegantrl',
                      env=env,
                      model_name='ppo',
                      if_vix=True,
                      API_KEY = API_KEY,
                      API_SECRET = API_SECRET,
                      API_BASE_URL = API_BASE_URL,
                      cwd='./papertrading_erl',
                      net_dimension = ERL_PARAMS['net_dimension'])

## Use full data to train

After tuning well, retrain on the training and testing sets

In [ ]:
train(start_date = '2022-08-25',
      end_date = '2022-09-02',
      ticker_list = ticker_list,
      data_source = 'alpaca',
      time_interval= '1Min',
      technical_indicator_list= INDICATORS,
      drl_lib='elegantrl',
      env=env,
      model_name='ppo',
      if_vix=True,
      API_KEY = API_KEY,
      API_SECRET = API_SECRET,
      API_BASE_URL = API_BASE_URL,
      erl_params=ERL_PARAMS,
      cwd='./papertrading_erl_retrain',
      break_step=2e5)

# Part 3: Deploy the agent

## Setup Alpaca Paper trading environment

In [ ]:
import datetime
import threading
from finrl.meta.data_processors.processor_alpaca import AlpacaProcessor
import alpaca_trade_api as tradeapi
import time
import pandas as pd
import numpy as np
import torch
import gym

class AlpacaPaperTrading():

    def __init__(self,ticker_list, time_interval, drl_lib, agent, cwd, net_dim,
                 state_dim, action_dim, API_KEY, API_SECRET,
                 API_BASE_URL, tech_indicator_list, turbulence_thresh=30,
                 max_stock=1e2, latency = None):
        #load agent
        self.drl_lib = drl_lib
        if agent =='ppo':
            if drl_lib == 'elegantrl':
                agent_class = AgentPPO
                agent = agent_class(net_dim, state_dim, action_dim)
                actor = agent.act
                # load agent
                try:
                    cwd = cwd + '/actor.pth'
                    print(f"| load actor from: {cwd}")
                    actor.load_state_dict(torch.load(cwd, map_location=lambda storage, loc: storage))
                    self.act = actor
                    self.device = agent.device
                except BaseException:
                    raise ValueError("Fail to load agent!")

            elif drl_lib == 'rllib':
                from ray.rllib.agents import ppo
                from ray.rllib.agents.ppo.ppo import PPOTrainer

                config = ppo.DEFAULT_CONFIG.copy()
                config['env'] = StockEnvEmpty
                config["log_level"] = "WARN"
                config['env_config'] = {'state_dim':state_dim,
                            'action_dim':action_dim,}
                trainer = PPOTrainer(env=StockEnvEmpty, config=config)
                trainer.restore(cwd)
                try:
                    trainer.restore(cwd)
                    self.agent = trainer
                    print("Restoring from checkpoint path", cwd)
                except:
                    raise ValueError('Fail to load agent!')

            elif drl_lib == 'stable_baselines3':
                from stable_baselines3 import PPO

                try:
                    #load agent
                    self.model = PPO.load(cwd)
                    print("Successfully load model", cwd)
                except:
                    raise ValueError('Fail to load agent!')

            else:
                raise ValueError('The DRL library input is NOT supported yet. Please check your input.')

        else:
            raise ValueError('Agent input is NOT supported yet.')



        #connect to Alpaca trading API
        try:
            self.alpaca = tradeapi.REST(API_KEY,API_SECRET,API_BASE_URL, 'v2')
        except:
            raise ValueError('Fail to connect Alpaca. Please check account info and internet connection.')

        #read trading time interval
        if time_interval == '1s':
            self.time_interval = 1
        elif time_interval == '5s':
            self.time_interval = 5
        elif time_interval == '1Min':
            self.time_interval = 60
        elif time_interval == '5Min':
            self.time_interval = 60 * 5
        elif time_interval == '15Min':
            self.time_interval = 60 * 15
        else:
            raise ValueError('Time interval input is NOT supported yet.')

        #read trading settings
        self.tech_indicator_list = tech_indicator_list
        self.turbulence_thresh = turbulence_thresh
        self.max_stock = max_stock

        #initialize account
        self.stocks = np.asarray([0] * len(ticker_list)) #stocks holding
        self.stocks_cd = np.zeros_like(self.stocks)
        self.cash = None #cash record
        self.stocks_df = pd.DataFrame(self.stocks, columns=['stocks'], index = ticker_list)
        self.asset_list = []
        self.price = np.asarray([0] * len(ticker_list))
        self.stockUniverse = ticker_list
        self.turbulence_bool = 0
        self.equities = []

    def test_latency(self, test_times = 10):
        total_time = 0
        for i in range(0, test_times):
            time0 = time.time()
            self.get_state()
            time1 = time.time()
            temp_time = time1 - time0
            total_time += temp_time
        latency = total_time/test_times
        print('latency for data processing: ', latency)
        return latency

    def run(self):
        orders = self.alpaca.list_orders(status="open")
        for order in orders:
          self.alpaca.cancel_order(order.id)

        # Wait for market to open.
        print("Waiting for market to open...")
        tAMO = threading.Thread(target=self.awaitMarketOpen)
        tAMO.start()
        tAMO.join()
        print("Market opened.")
        while True:

          # Figure out when the market will close so we can prepare to sell beforehand.
          clock = self.alpaca.get_clock()
          closingTime = clock.next_close.replace(tzinfo=datetime.timezone.utc).timestamp()
          currTime = clock.timestamp.replace(tzinfo=datetime.timezone.utc).timestamp()
          self.timeToClose = closingTime - currTime

          if(self.timeToClose < (60)):
            # Close all positions when 1 minutes til market close.
            print("Market closing soon. Stop trading.")
            break

            '''# Close all positions when 1 minutes til market close.
            print("Market closing soon.  Closing positions.")

            positions = self.alpaca.list_positions()
            for position in positions:
              if(position.side == 'long'):
                orderSide = 'sell'
              else:
                orderSide = 'buy'
              qty = abs(int(float(position.qty)))
              respSO = []
              tSubmitOrder = threading.Thread(target=self.submitOrder(qty, position.symbol, orderSide, respSO))
              tSubmitOrder.start()
              tSubmitOrder.join()

            # Run script again after market close for next trading day.
            print("Sleeping until market close (15 minutes).")
            time.sleep(60 * 15)'''

          else:
            trade = threading.Thread(target=self.trade)
            trade.start()
            trade.join()
            last_equity = float(self.alpaca.get_account().last_equity)
            cur_time = time.time()
            self.equities.append([cur_time,last_equity])
            time.sleep(self.time_interval)

    def awaitMarketOpen(self):
        isOpen = self.alpaca.get_clock().is_open
        while(not isOpen):
          clock = self.alpaca.get_clock()
          openingTime = clock.next_open.replace(tzinfo=datetime.timezone.utc).timestamp()
          currTime = clock.timestamp.replace(tzinfo=datetime.timezone.utc).timestamp()
          timeToOpen = int((openingTime - currTime) / 60)
          print(str(timeToOpen) + " minutes til market open.")
          time.sleep(60)
          isOpen = self.alpaca.get_clock().is_open

    def trade(self):
        state = self.get_state()

        if self.drl_lib == 'elegantrl':
            with torch.no_grad():
                s_tensor = torch.as_tensor((state,), device=self.device)
                a_tensor = self.act(s_tensor)
                action = a_tensor.detach().cpu().numpy()[0]
            action = (action * self.max_stock).astype(int)

        elif self.drl_lib == 'rllib':
            action = self.agent.compute_single_action(state)

        elif self.drl_lib == 'stable_baselines3':
            action = self.model.predict(state)[0]

        else:
            raise ValueError('The DRL library input is NOT supported yet. Please check your input.')

        self.stocks_cd += 1
        if self.turbulence_bool == 0:
            min_action = 10  # stock_cd
            for index in np.where(action < -min_action)[0]:  # sell_index:
                sell_num_shares = min(self.stocks[index], -action[index])
                qty =  abs(int(sell_num_shares))
                respSO = []
                tSubmitOrder = threading.Thread(target=self.submitOrder(qty, self.stockUniverse[index], 'sell', respSO))
                tSubmitOrder.start()
                tSubmitOrder.join()
                self.cash = float(self.alpaca.get_account().cash)
                self.stocks_cd[index] = 0

            for index in np.where(action > min_action)[0]:  # buy_index:
                if self.cash < 0:
                    tmp_cash = 0
                else:
                    tmp_cash = self.cash
                buy_num_shares = min(tmp_cash // self.price[index], abs(int(action[index])))
                if (buy_num_shares != buy_num_shares): # if buy_num_change = nan
                    qty = 0 # set to 0 quantity
                else:
                    qty = abs(int(buy_num_shares))
                qty = abs(int(buy_num_shares))
                respSO = []
                tSubmitOrder = threading.Thread(target=self.submitOrder(qty, self.stockUniverse[index], 'buy', respSO))
                tSubmitOrder.start()
                tSubmitOrder.join()
                self.cash = float(self.alpaca.get_account().cash)
                self.stocks_cd[index] = 0

        else:  # sell all when turbulence
            positions = self.alpaca.list_positions()
            for position in positions:
                if(position.side == 'long'):
                    orderSide = 'sell'
                else:
                    orderSide = 'buy'
                qty = abs(int(float(position.qty)))
                respSO = []
                tSubmitOrder = threading.Thread(target=self.submitOrder(qty, position.symbol, orderSide, respSO))
                tSubmitOrder.start()
                tSubmitOrder.join()

            self.stocks_cd[:] = 0


    def get_state(self):
        alpaca = AlpacaProcessor(api=self.alpaca)
        price, tech, turbulence = alpaca.fetch_latest_data(ticker_list = self.stockUniverse, time_interval='1Min',
                                                     tech_indicator_list=self.tech_indicator_list)
        turbulence_bool = 1 if turbulence >= self.turbulence_thresh else 0

        turbulence = (self.sigmoid_sign(turbulence, self.turbulence_thresh) * 2 ** -5).astype(np.float32)

        tech = tech * 2 ** -7
        positions = self.alpaca.list_positions()
        stocks = [0] * len(self.stockUniverse)
        for position in positions:
            ind = self.stockUniverse.index(position.symbol)
            stocks[ind] = ( abs(int(float(position.qty))))

        stocks = np.asarray(stocks, dtype = float)
        cash = float(self.alpaca.get_account().cash)
        self.cash = cash
        self.stocks = stocks
        self.turbulence_bool = turbulence_bool
        self.price = price



        amount = np.array(self.cash * (2 ** -12), dtype=np.float32)
        scale = np.array(2 ** -6, dtype=np.float32)
        state = np.hstack((amount,
                    turbulence,
                    self.turbulence_bool,
                    price * scale,
                    self.stocks * scale,
                    self.stocks_cd,
                    tech,
                    )).astype(np.float32)
        state[np.isnan(state)] = 0.0
        state[np.isinf(state)] = 0.0
        print(len(self.stockUniverse))
        return state

    def submitOrder(self, qty, stock, side, resp):
        if(qty > 0):
          try:
            self.alpaca.submit_order(stock, qty, side, "market", "day")
            print("Market order of | " + str(qty) + " " + stock + " " + side + " | completed.")
            resp.append(True)
          except:
            print("Order of | " + str(qty) + " " + stock + " " + side + " | did not go through.")
            resp.append(False)
        else:
          print("Quantity is 0, order of | " + str(qty) + " " + stock + " " + side + " | not completed.")
          resp.append(True)

    @staticmethod
    def sigmoid_sign(ary, thresh):
        def sigmoid(x):
            return 1 / (1 + np.exp(-x * np.e)) - 0.5

        return sigmoid(ary / thresh) * thresh

class StockEnvEmpty(gym.Env):
    #Empty Env used for loading rllib agent
    def __init__(self,config):
      state_dim = config['state_dim']
      action_dim = config['action_dim']
      self.env_num = 1
      self.max_step = 10000
      self.env_name = 'StockEnvEmpty'
      self.state_dim = state_dim
      self.action_dim = action_dim
      self.if_discrete = False
      self.target_return = 9999
      self.observation_space = gym.spaces.Box(low=-3000, high=3000, shape=(state_dim,), dtype=np.float32)
      self.action_space = gym.spaces.Box(low=-1, high=1, shape=(action_dim,), dtype=np.float32)

    def reset(self):
        return

    def step(self, actions):
        return

## Run Paper trading

In [ ]:
print(DOW_30_TICKER)

In [ ]:
state_dim

In [ ]:
action_dim

In [ ]:
paper_trading_erl = AlpacaPaperTrading(ticker_list = DOW_30_TICKER,
                                       time_interval = '1Min',
                                       drl_lib = 'elegantrl',
                                       agent = 'ppo',
                                       cwd = './papertrading_erl_retrain',
                                       net_dim = ERL_PARAMS['net_dimension'],
                                       state_dim = state_dim,
                                       action_dim= action_dim,
                                       API_KEY = API_KEY,
                                       API_SECRET = API_SECRET,
                                       API_BASE_URL = API_BASE_URL,
                                       tech_indicator_list = INDICATORS,
                                       turbulence_thresh=30,
                                       max_stock=1e2)
paper_trading_erl.run()

# Part 4: Check Portfolio Performance

In [ ]:
import alpaca_trade_api as tradeapi
import pandas_market_calendars as tc
import numpy as np
import pandas as pd
import pytz
import yfinance as yf
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from datetime import datetime as dt
from finrl.plot import backtest_stats
import matplotlib.pyplot as plt

In [ ]:
def get_trading_days(start, end):
    nyse = tc.get_calendar('NYSE')
    df = nyse.date_range_htf("1D", pd.Timestamp(start),
                                pd.Timestamp(end))
    trading_days = []
    for day in df:
        trading_days.append(str(day)[:10])

    return trading_days

def alpaca_history(key, secret, url, start, end):
    api = tradeapi.REST(key, secret, url, 'v2')
    trading_days = get_trading_days(start, end)
    df = pd.DataFrame()
    for day in trading_days:
        df = pd.concat([df, api.get_portfolio_history(date_start = day,timeframe='5Min').df.iloc[:78]])
    equities = df.equity.values
    cumu_returns = equities/equities[0]
    cumu_returns = cumu_returns[~np.isnan(cumu_returns)]

    return df, cumu_returns

def DIA_history(start):
    data_df = yf.download(['^DJI'],start=start, interval="5m")
    data_df = data_df.iloc[:]
    baseline_returns = data_df['Adj Close'].values/data_df['Adj Close'].values[0]
    return data_df, baseline_returns

## Get cumulative return

In [ ]:
API_KEY = ""
API_SECRET = ""
API_BASE_URL = 'https://paper-api.alpaca.markets'
data_url = 'wss://data.alpaca.markets'

In [ ]:
df_erl, cumu_erl = alpaca_history(key=API_KEY,
                                  secret=API_SECRET,
                                  url=API_BASE_URL,
                                  start='2022-09-01', #must be within 1 month
                                  end='2022-09-12') #change the date if error occurs


In [ ]:
df_djia, cumu_djia = DIA_history(start='2022-09-01')

In [ ]:
df_erl.tail()

In [ ]:
returns_erl = cumu_erl -1
returns_dia = cumu_djia - 1
returns_dia = returns_dia[:returns_erl.shape[0]]
print('len of erl return: ', returns_erl.shape[0])
print('len of dia return: ', returns_dia.shape[0])

In [ ]:
returns_erl

## plot and save

In [ ]:
import matplotlib.pyplot as plt
plt.figure(dpi=1000)
plt.grid()
plt.grid(which='minor', axis='y')
plt.title('Stock Trading (Paper trading)', fontsize=20)
plt.plot(returns_erl, label = 'ElegantRL Agent', color = 'red')
#plt.plot(returns_sb3, label = 'Stable-Baselines3 Agent', color = 'blue' )
#plt.plot(returns_rllib, label = 'RLlib Agent', color = 'green')
plt.plot(returns_dia, label = 'DJIA', color = 'grey')
plt.ylabel('Return', fontsize=16)
plt.xlabel('Year 2021', fontsize=16)
plt.xticks(size = 14)
plt.yticks(size = 14)
ax = plt.gca()
ax.xaxis.set_major_locator(ticker.MultipleLocator(78))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.005))
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=2))
ax.xaxis.set_major_formatter(ticker.FixedFormatter(['','10-19','','10-20',
                                                    '','10-21','','10-22']))
plt.legend(fontsize=10.5)
plt.savefig('papertrading_stock.png')